# Chapter 6: Image Compression with SVD

This notebook demonstrates how Singular Value Decomposition can be used for image compression:

1. Image Compression with SVD
2. Rank-k Approximations
3. Visualization of Compression Quality vs Rank

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import ipywidgets as widgets
from ipywidgets import interact, interactive

# Configure matplotlib
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 1. Understanding SVD for Images

For an image matrix $A$ of size $m \times n$:

$$A = U \Sigma V^T$$

We can approximate $A$ using only the top $k$ singular values:

$$A_k = U_k \Sigma_k V_k^T$$

This is the **best rank-k approximation** in terms of Frobenius norm.

In [ ]:
# Create a synthetic test image with various features
def create_test_image(size=256):
    """
    Create a test image with gradients, edges, and patterns.
    """
    img = np.zeros((size, size))
    
    # Add horizontal gradient
    for i in range(size):
        img[:, i] += i / size * 0.3
    
    # Add vertical gradient
    for i in range(size):
        img[i, :] += i / size * 0.2
    
    # Add circles
    y, x = np.ogrid[:size, :size]
    center1 = (size//3, size//3)
    center2 = (2*size//3, 2*size//3)
    
    r1 = np.sqrt((x - center1[1])**2 + (y - center1[0])**2)
    r2 = np.sqrt((x - center2[1])**2 + (y - center2[0])**2)
    
    img += 0.3 * (r1 < size//6)
    img += 0.4 * np.exp(-r2**2 / (2*(size//8)**2))
    
    # Add a rectangle
    img[size//4:size//2, size//2:3*size//4] += 0.3
    
    # Add diagonal stripes
    for i in range(0, size, 20):
        for j in range(5):
            if i+j < size:
                np.fill_diagonal(img[i+j:, :], 0.2 * (j < 3))
                np.fill_diagonal(img[:, i+j:], 0.2 * (j < 3))
    
    # Normalize to [0, 1]
    img = (img - img.min()) / (img.max() - img.min())
    
    return img

# Create and display the image
original_image = create_test_image(256)

plt.figure(figsize=(8, 8))
plt.imshow(original_image, cmap='gray')
plt.colorbar()
plt.title(f'Original Image ({original_image.shape[0]} x {original_image.shape[1]})')
plt.axis('off')
plt.show()

print(f"Image shape: {original_image.shape}")
print(f"Total pixels: {original_image.size}")

In [ ]:
# Compute SVD of the image
U, S, Vt = np.linalg.svd(original_image, full_matrices=False)

print(f"U shape: {U.shape}")
print(f"S shape: {S.shape}")
print(f"Vt shape: {Vt.shape}")

# Visualize singular value spectrum
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax = axes[0]
ax.plot(S, 'b.-', markersize=4)
ax.set_xlabel('Index')
ax.set_ylabel('Singular Value')
ax.set_title('Singular Values (Linear Scale)')
ax.grid(True, alpha=0.3)

# Log scale
ax = axes[1]
ax.semilogy(S, 'b.-', markersize=4)
ax.set_xlabel('Index')
ax.set_ylabel('Singular Value (log)')
ax.set_title('Singular Values (Log Scale)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Cumulative energy
energy = np.cumsum(S**2) / np.sum(S**2)
print(f"\nEnergy captured by top k singular values:")
for k in [1, 5, 10, 20, 50, 100]:
    if k <= len(energy):
        print(f"  k={k:3d}: {energy[k-1]*100:.2f}%")

## 2. Rank-k Approximations

Let's implement and visualize low-rank approximations.

In [ ]:
def svd_compress(image, k):
    """
    Compress an image using rank-k SVD approximation.
    
    Parameters:
        image: 2D numpy array (grayscale image)
        k: number of singular values to keep
    
    Returns:
        compressed: rank-k approximation of the image
        compression_ratio: ratio of original to compressed size
    """
    U, S, Vt = np.linalg.svd(image, full_matrices=False)
    
    # Keep only top k components
    U_k = U[:, :k]
    S_k = S[:k]
    Vt_k = Vt[:k, :]
    
    # Reconstruct
    compressed = U_k @ np.diag(S_k) @ Vt_k
    
    # Calculate compression ratio
    m, n = image.shape
    original_size = m * n
    compressed_size = k * (m + n + 1)  # U_k + S_k + Vt_k
    compression_ratio = original_size / compressed_size
    
    return compressed, compression_ratio

def compute_psnr(original, compressed):
    """Compute Peak Signal-to-Noise Ratio."""
    mse = np.mean((original - compressed)**2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    return 10 * np.log10(max_pixel**2 / mse)

# Compare different ranks
ranks = [1, 5, 10, 20, 50, 100, 200]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Original
axes[0].imshow(original_image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

# Compressed versions
for i, k in enumerate(ranks):
    compressed, ratio = svd_compress(original_image, k)
    psnr = compute_psnr(original_image, compressed)
    
    axes[i+1].imshow(compressed, cmap='gray', vmin=0, vmax=1)
    axes[i+1].set_title(f'k={k}\nRatio: {ratio:.1f}x, PSNR: {psnr:.1f}dB')
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Interactive compression visualization
def interactive_svd_compression(k=10):
    """
    Interactively explore SVD compression with different ranks.
    """
    compressed, ratio = svd_compress(original_image, k)
    diff = np.abs(original_image - compressed)
    psnr = compute_psnr(original_image, compressed)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original
    axes[0].imshow(original_image, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    # Compressed
    axes[1].imshow(compressed, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title(f'Compressed (k={k})\nCompression: {ratio:.1f}x')
    axes[1].axis('off')
    
    # Difference
    im = axes[2].imshow(diff, cmap='hot', vmin=0, vmax=0.3)
    axes[2].set_title(f'Absolute Error\nPSNR: {psnr:.1f} dB')
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    
    plt.tight_layout()
    plt.show()
    
    # Storage comparison
    m, n = original_image.shape
    original_size = m * n
    compressed_size = k * (m + n + 1)
    print(f"Storage comparison:")
    print(f"  Original: {original_size:,} values")
    print(f"  Compressed: {compressed_size:,} values")
    print(f"  Savings: {(1 - compressed_size/original_size)*100:.1f}%")

interact(interactive_svd_compression,
         k=widgets.IntSlider(min=1, max=min(original_image.shape), step=1, value=10,
                            description='Rank k'));

## 3. Compression Quality vs Rank Analysis

Let's systematically analyze how compression quality varies with the number of singular values.

In [ ]:
# Compute metrics for all ranks
max_rank = min(original_image.shape)
ranks_to_test = list(range(1, max_rank + 1))

compression_ratios = []
psnrs = []
errors = []

for k in ranks_to_test:
    compressed, ratio = svd_compress(original_image, k)
    psnr = compute_psnr(original_image, compressed)
    error = np.linalg.norm(original_image - compressed, 'fro')
    
    compression_ratios.append(ratio)
    psnrs.append(psnr)
    errors.append(error)

compression_ratios = np.array(compression_ratios)
psnrs = np.array(psnrs)
errors = np.array(errors)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PSNR vs Rank
ax = axes[0, 0]
ax.plot(ranks_to_test, psnrs, 'b-', linewidth=2)
ax.axhline(y=30, color='r', linestyle='--', label='30 dB (good quality)')
ax.axhline(y=40, color='g', linestyle='--', label='40 dB (excellent quality)')
ax.set_xlabel('Rank k')
ax.set_ylabel('PSNR (dB)')
ax.set_title('Reconstruction Quality vs Rank')
ax.legend()
ax.grid(True, alpha=0.3)

# Compression Ratio vs Rank
ax = axes[0, 1]
ax.semilogy(ranks_to_test, compression_ratios, 'g-', linewidth=2)
ax.set_xlabel('Rank k')
ax.set_ylabel('Compression Ratio (log scale)')
ax.set_title('Compression Ratio vs Rank')
ax.grid(True, alpha=0.3)

# Error vs Rank
ax = axes[1, 0]
ax.semilogy(ranks_to_test, errors, 'r-', linewidth=2)
ax.set_xlabel('Rank k')
ax.set_ylabel('Frobenius Error (log scale)')
ax.set_title('Reconstruction Error vs Rank')
ax.grid(True, alpha=0.3)

# Quality-Compression Trade-off
ax = axes[1, 1]
sc = ax.scatter(compression_ratios, psnrs, c=ranks_to_test, cmap='viridis', 
                s=20, alpha=0.7)
ax.set_xlabel('Compression Ratio')
ax.set_ylabel('PSNR (dB)')
ax.set_title('Quality-Compression Trade-off')
ax.set_xscale('log')
plt.colorbar(sc, ax=ax, label='Rank k')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Find optimal rank for different quality targets
quality_targets = [25, 30, 35, 40, 45]

print("Optimal rank for different quality targets:")
print("-" * 50)
print(f"{'Target PSNR':>12} | {'Min Rank':>10} | {'Compression':>12} | {'Savings':>10}")
print("-" * 50)

for target in quality_targets:
    # Find minimum rank that achieves target PSNR
    valid_ranks = np.where(psnrs >= target)[0]
    if len(valid_ranks) > 0:
        min_rank = valid_ranks[0] + 1
        ratio = compression_ratios[min_rank - 1]
        savings = (1 - 1/ratio) * 100
        print(f"{target:>10} dB | {min_rank:>10} | {ratio:>10.1f}x | {savings:>9.1f}%")
    else:
        print(f"{target:>10} dB | Not achievable")

In [ ]:
# Extension: Color image compression
def create_color_test_image(size=256):
    """Create a color test image."""
    img = np.zeros((size, size, 3))
    
    y, x = np.ogrid[:size, :size]
    
    # Red channel: horizontal gradient
    img[:, :, 0] = x / size
    
    # Green channel: vertical gradient
    img[:, :, 1] = y / size
    
    # Blue channel: circular pattern
    center = size // 2
    r = np.sqrt((x - center)**2 + (y - center)**2)
    img[:, :, 2] = 0.5 + 0.5 * np.sin(r / 10)
    
    # Add some shapes
    img[size//4:size//2, size//4:size//2, 0] = 1.0
    img[size//2:3*size//4, size//2:3*size//4, 1] = 1.0
    
    return np.clip(img, 0, 1)

def svd_compress_color(image, k):
    """Compress a color image by applying SVD to each channel."""
    compressed = np.zeros_like(image)
    
    for c in range(3):
        U, S, Vt = np.linalg.svd(image[:, :, c], full_matrices=False)
        compressed[:, :, c] = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    
    return np.clip(compressed, 0, 1)

# Create and compress color image
color_image = create_color_test_image(256)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Original
axes[0, 0].imshow(color_image)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

# Compressed versions
color_ranks = [1, 5, 10, 20, 50, 100, 200]
for i, k in enumerate(color_ranks):
    ax = axes.flatten()[i+1]
    compressed = svd_compress_color(color_image, k)
    ax.imshow(compressed)
    ax.set_title(f'k={k}')
    ax.axis('off')

plt.suptitle('Color Image SVD Compression', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize individual SVD components
U, S, Vt = np.linalg.svd(original_image, full_matrices=False)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for i in range(15):
    ax = axes.flatten()[i]
    
    # Single component: sigma_i * u_i * v_i^T
    component = S[i] * np.outer(U[:, i], Vt[i, :])
    
    im = ax.imshow(component, cmap='RdBu', vmin=-0.5, vmax=0.5)
    ax.set_title(f'Component {i+1}\n$\\sigma_{{{i+1}}}$={S[i]:.2f}')
    ax.axis('off')

plt.suptitle('First 15 SVD Components', fontsize=14)
plt.tight_layout()
plt.show()

print("Each component captures different spatial frequencies:")
print("- Early components: large-scale structure, smooth gradients")
print("- Later components: fine details, edges, noise")

In [ ]:
# Progressive reconstruction animation (static version)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

progressive_ranks = [1, 2, 5, 10, 20, 30, 50, 75, 100, 150]

for i, k in enumerate(progressive_ranks):
    ax = axes.flatten()[i]
    compressed, ratio = svd_compress(original_image, k)
    ax.imshow(compressed, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'k={k}')
    ax.axis('off')

plt.suptitle('Progressive SVD Reconstruction', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

In this notebook, we explored SVD-based image compression:

1. **SVD Decomposition**: Any image matrix can be factored into $U\Sigma V^T$
2. **Rank-k Approximation**: Keeping top $k$ singular values gives the best rank-$k$ approximation
3. **Quality vs Compression Trade-off**: Higher rank = better quality but less compression

### Key Insights:

- Singular values decay rapidly for natural images
- 90%+ compression is often achievable with acceptable quality
- Early components capture global structure, later ones capture details
- SVD is optimal for Frobenius norm but not perceptual quality

### Practical Applications:

- Image compression (though JPEG uses DCT instead)
- Noise reduction (truncate small singular values)
- Latent space representations in neural networks
- Matrix approximation in recommendation systems

---

*Continue to Chapter 11 for gradient descent deep dive...*